# Fase 0 - Ingesta de datos

In [1]:
import os
import urllib

In [2]:
if not os.path.exists("data"):
    os.makedirs("data")

## Descargar Datasets Abiertos

source: https://developer.imdb.com/non-commercial-datasets/

In [3]:
urllib.request.urlretrieve(
    "https://datasets.imdbws.com/title.basics.tsv.gz", 
    "data/title.basics.tsv.gz"
)
urllib.request.urlretrieve(
    "https://datasets.imdbws.com/title.ratings.tsv.gz", 
    "data/title.ratings.tsv.gz"
) 

('data/title.ratings.tsv.gz', <http.client.HTTPMessage at 0xffff993fcb30>)

In [12]:
# !uv add polars

Resolved 3 packages in 1ms
Checked 2 packages in 0.67ms


In [6]:
import polars as pl
pl.__version__


'1.42.1'

### Basics Data

In [8]:
basics_df = pl.read_csv(
    "data/title.basics.tsv.gz",
    separator="\t",
    null_values="\\N", # to handle missing values
    quote_char=None,   # to avoid issues with quotes in the data
)

In [9]:
basics_df.head()

tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
str,str,str,str,i64,i64,str,i64,str
"""tt0000001""","""short""","""Carmencita""","""Carmencita""",0,1894,null,1,"""Documentary,Short"""
"""tt0000002""","""short""","""Le clown et ses chiens""","""Le clown et ses chiens""",0,1892,null,5,"""Animation,Short"""
"""tt0000003""","""short""","""Poor Pierrot""","""Pauvre Pierrot""",0,1892,null,5,"""Animation,Comedy,Romance"""
"""tt0000004""","""short""","""Un bon bock""","""Un bon bock""",0,1892,null,12,"""Animation,Short"""
"""tt0000005""","""short""","""Blacksmith Scene""","""Blacksmith Scene""",0,1893,null,1,"""Short"""


In [10]:
# how many rows and columns are in the dataframe
basics_df.shape

(12633670, 9)

### Ratings Data

In [11]:
ratings_df = pl.read_csv(
    "data/title.ratings.tsv.gz",
    separator="\t",
    null_values="\\N", # to handle missing values
    quote_char=None,   # to avoid issues with quotes in the data
)
ratings_df.head()

tconst,averageRating,numVotes
str,f64,i64
"""tt0000001""",5.7,2216
"""tt0000002""",5.4,321
"""tt0000003""",6.4,2354
"""tt0000004""",5.1,193
"""tt0000005""",6.2,3069


In [14]:
ratings_df.shape

(1693823, 3)

## Unificar Datasets Abiertos

In [15]:
# merge the two dataframes on the tconst column, which is the unique identifier for each title in the IMDB dataset
movies_df = (
    basics_df
    .filter(pl.col("titleType") == "movie")  # filter for movies only
    .select([
        pl.col("tconst").alias("imdb_movie_id"), # rename the column to imdb_movie_id
        pl.col("primaryTitle").alias("title"), # rename the column to title
        pl.col("startYear").cast(pl.Int32).alias("year"), # rename the column to year and cast it to Int32
        pl.col("genres") # keep the genres column
    ])
    .join(
        ratings_df.select(["tconst", "averageRating", "numVotes"]), # select only the necessary columns from ratings_df
        left_on="imdb_movie_id",    # join on the imdb_movie_id column from basics_df
        right_on="tconst"           # join on the tconst column from ratings_df
    )
    .rename({       # rename columns in the merged dataframe
        "averageRating": "imdb_rating", # rename the column to imdb_rating
        "numVotes": "imdb_votes"        # rename the column to imdb_votes
    })
    .filter(
        (pl.col("imdb_votes") >= 1000)   # filter for movies with at least 1000 votes
    )
)

In [16]:
movies_df.head()

imdb_movie_id,title,year,genres,imdb_rating,imdb_votes
str,str,i32,str,f64,i64
"""tt0000574""","""The Story of the Kelly Gang""",1906,"""Action,Adventure,Biography""",6.0,1076
"""tt0002130""","""Dante's Inferno""",1911,"""Adventure,Drama,Fantasy""",7.1,4176
"""tt0002423""","""Passion""",1919,"""Biography,Drama,Romance""",6.7,1132
"""tt0002844""","""Fantômas: In the Shadow of the…",1913,"""Crime,Drama""",6.9,2738
"""tt0003014""","""Ingeborg Holm""",1913,"""Drama""",7.0,1614


In [17]:
movies_df.write_parquet("data/movies.parquet", compression="snappy")

In [18]:
movies_df.shape

(48692, 6)

## Guardar Ids de películas

In [19]:
# write the imdb_movie_id column to a text file, one id per line
with open("data/movie_ids.txt", "w") as f:
    for movie_id in movies_df["imdb_movie_id"].to_list():
        f.write(f"{movie_id}\n")

In [20]:
!head data/movie_ids.txt

tt0000574
tt0002130
tt0002423
tt0002844
tt0003014
tt0003037
tt0003165
tt0003419
tt0003643
tt0003740


## Descargar Dataset Privado de OMDB API

source: https://www.omdbapi.com/

In [2]:
# !uv add dotenv loguru aiohttp orjson tqdm

Resolved 20 packages in 0.84ms
Checked 17 packages in 2ms


In [5]:
from dotenv import load_dotenv
from loguru import logger
from tqdm import tqdm
from pathlib import Path

import aiohttp
import asyncio

import orjson

from typing import List, Dict

In [6]:
import os

# Cargar variables de entorno desde archivo .env (incluida la API KEY)
load_dotenv()

OMDB_API_KEY = os.getenv("OMDB_API_KEY")

In [9]:
# ========== CONFIGURACIÓN GLOBAL ==========
# URL base de la API de OMDb
OMDB_API_URL = "https://www.omdbapi.com/"
# Número de workers concurrentes para descargar datos en paralelo
# CONCURRENCY = 20
CONCURRENCY = 5
# Número máximo de reintentos por cada ID que falla
MAX_RETRIES = 5
# Ruta del archivo para guardar qué IDs ya fueron procesados
CHECKPOINT_FILE = "data/processed_ids.txt"


# ========== CONFIGURACIÓN DE LOGURU ==========
# Eliminar los manejadores de logs por defecto
logger.remove()
# Agregar un nuevo manejador de logs con formato personalizado y colores
logger.add(
    lambda msg: tqdm.write(msg, end=""),  # Función que imprime el mensaje usando tqdm para no romper la barra de progreso
    # lambda msg: print(msg, end=""),  # Función que imprime el mensaje sin agregar salto de línea extra
    colorize=True,  # Habilitar colores en los logs
    format="<green>{time:HH:mm:ss}</green> | "  # Hora en verde
           "<level>{level}</level> | "  # Nivel de log (INFO, WARNING, etc.)
           "{message}\n",  # Mensaje del log
)


# ========== UTILIDADES DE CHECKPOINT ==========
# Función para cargar IDs que ya fueron procesados
def load_processed_ids() -> set[str]:
    """Carga los IDs de IMDb ya procesados desde el archivo de checkpoint."""
    # Verificar si el archivo de checkpoint existe
    if Path(CHECKPOINT_FILE).exists():
        logger.info(f"Loading checkpoint from {CHECKPOINT_FILE}")
        # Abrir el archivo en modo lectura
        with open(CHECKPOINT_FILE, "r") as f:
            # Leer cada línea, eliminar espacios en blanco, y crear un conjunto
            return set(line.strip() for line in f if line.strip())
    # Si el archivo no existe, retornar un conjunto vacío
    return set()


# Función para guardar un ID procesado en el checkpoint
def append_processed_id(imdb_id: str) -> None:
    """Agrega un único ID de IMDb al archivo de checkpoint."""
    # Abrir el archivo en modo append (añadir al final)
    with open(CHECKPOINT_FILE, "a") as f:
        # Escribir el ID seguido de un salto de línea
        f.write(imdb_id + "\n")


# ========== FUNCIÓN PARA UN ÚNICO REQUEST A OMDb ==========
# Función asíncrona para obtener datos de una película de OMDb
# async def fetch_one(session, imdb_id):
async def fetch_one(session: aiohttp.ClientSession, imdb_id: str) -> Dict | None:
    """
    Obtiene un registro de OMDb por su ID de IMDb.
    Retorna: {"imdb_id": ..., "raw": <json_de_omdb>} o None si falla permanentemente.
    """
    # Definir los parámetros para el request a OMDb
    params = {
        "i": imdb_id,  # ID de IMDb (ej: tt1234567)
        "apikey": os.getenv("OMDB_API_KEY"),  # API key desde variables de entorno
        "plot": "full",  # Obtener la trama completa de la película
    }
    # Inicializar contador de reintentos
    retries = 0
    # logger.info(params)

    # Bucle que se ejecuta mientras no se alcance el máximo de reintentos
    while retries < MAX_RETRIES:
        try:
            # logger.info(f"[{imdb_id}] AFTER request")
            # Hacer un request GET asíncrono a la API
            async with session.get(OMDB_API_URL, params=params) as resp:
                # logger.info(f"[{imdb_id}] BEFORE request")
                # Obtener el código de estado HTTP
                status = resp.status

                # Manejar error 429: Too Many Requests (limite de rate)
                if status == 429:
                    # Calcular tiempo de espera: 5 segundos por reintento
                    wait = 5 * (retries + 1)
                    logger.warning(f"[{imdb_id}] 429 Too Many Requests — sleeping {wait}s")
                    # Esperar antes de reintentar
                    await asyncio.sleep(wait)
                    # Incrementar contador de reintentos
                    retries += 1
                    # Saltar a la siguiente iteración del bucle
                    continue

                # Manejar errores del servidor (5xx)
                if status >= 500:
                    # Calcular tiempo de espera: 3 segundos por reintento
                    wait = 3 * (retries + 1)
                    logger.warning(f"[{imdb_id}] Server error {status} — retrying in {wait}s")
                    # Esperar antes de reintentar
                    await asyncio.sleep(wait)
                    # Incrementar contador de reintentos
                    retries += 1
                    # Saltar a la siguiente iteración del bucle
                    continue

                # Convertir la respuesta a JSON
                data = await resp.json()

                # Verificar si OMDb retornó un error lógico (ID inválido, etc.)
                if data.get("Response") == "False":
                    # Obtener el mensaje de error de OMDb
                    err = data.get("Error", "Unknown error")
                    logger.error(f"[{imdb_id}] OMDb error: {err}")
                    # Retornar el resultado con el error (para guardar en log)
                    return {"imdb_id": imdb_id, "raw": data}

                # Si todo fue exitoso, retornar los datos (sin generar log para evitar spam)
                return {"imdb_id": imdb_id, "raw": data}

        # Manejar excepción de timeout
        except asyncio.TimeoutError:
            # Incrementar contador de reintentos
            retries += 1
            # Calcular tiempo de espera: 2 segundos por reintento
            wait = 2 * retries
            logger.warning(f"[{imdb_id}] Timeout — retrying in {wait}s")
            # Esperar antes de reintentar
            await asyncio.sleep(wait)

        # Manejar cualquier otra excepción inesperada
        except Exception as e:
            # Incrementar contador de reintentos
            retries += 1
            # Calcular tiempo de espera: 2 segundos por reintento
            wait = 2 * retries
            logger.error(f"[{imdb_id}] Unexpected error: {e} — retrying in {wait}s")
            # Esperar antes de reintentar
            await asyncio.sleep(wait)

    # Si se agotan todos los reintentos, registrar error crítico
    logger.critical(f"[{imdb_id}] FAILED permanently after {MAX_RETRIES} retries")
    # Retornar None para indicar fallo definitivo
    return None


# ========== WORKER: PROCESA IDs DE LA COLA Y ESCRIBE A JSONL ==========
# Función asíncrona que actúa como worker para procesar IDs en paralelo
async def worker(
    worker_id: int,  # Identificador del worker
    session: aiohttp.ClientSession,  # Sesión HTTP compartida
    queue: asyncio.Queue,  # Cola de IDs a procesar
    jsonl_path: Path,  # Ruta del archivo JSONL para guardar resultados
    write_lock: asyncio.Lock,  # Lock para escribir de forma segura en el archivo
    progress: tqdm # Barra de progreso opcional (para mostrar progreso en consola)
):
    # Bucle infinito que procesa IDs de la cola
    while True:
        # Obtener el siguiente ID de la cola (espera si está vacía)
        imdb_id = await queue.get()
        # Verificar si es la señal de terminación (None)
        if imdb_id is None:
            # Marcar la tarea como completada
            queue.task_done()
            # Salir del bucle para terminar el worker
            return

        logger.info(f"Worker {worker_id}: {imdb_id}")

        # Obtener datos de OMDb para este ID
        result = await fetch_one(session, imdb_id)

        # Verificar si el fetch fue exitoso
        if result is not None:
            # Convertir el resultado a formato JSON y luego a string (usando orjson)
            line = orjson.dumps(result).decode("utf-8") + "\n"

            # Usar un lock para asegurar que solo un worker escriba a la vez
            async with write_lock:
                # Abrir el archivo JSONL en modo append
                with open(jsonl_path, "a", encoding="utf-8") as f:
                    # Escribir una línea JSON
                    f.write(line)

            # Guardar el ID como procesado en el checkpoint
            append_processed_id(imdb_id)
        
        # Actualizar la barra de progreso
        progress.update(1)

        # Marcar la tarea como completada en la cola
        queue.task_done()


# ========== FUNCIÓN PRINCIPAL: PIPELINE ASÍNCRONO ==========
# Función asíncrona principal que orquesta todo el descargador
async def main(
    all_ids: List[str],  # Lista de todos los IDs de IMDb a descargar
    output_file: str = "omdb_raw.jsonl",  # Ruta del archivo de salida JSONL
):
    """
    Punto de entrada principal del descargador de OMDb.
    
    - all_ids: lista de IDs de IMDb (formato: ttXXXXXX)
    - output_file: ruta del archivo JSONL para guardar los resultados
    
    Esta función:
    - Salta los IDs que ya fueron procesados
    - Descarga los IDs restantes con concurrencia
    - Guarda cada resultado como una línea JSON en output_file
    - Es segura para reintentar después de crashes
    """
    # Convertir el path a un objeto Path
    jsonl_path = Path(output_file)

    # Cargar los IDs que ya fueron procesados desde el checkpoint
    processed = load_processed_ids()
    # Crear una lista de IDs que aún no han sido procesados
    remaining = [i for i in all_ids if i not in processed]
    # Crear una barra de progreso para mostrar el avance de la descarga
    progress = tqdm(
        total=len(remaining),
        desc="Downloading OMDb",
        unit="movie",
        colour="green",
        dynamic_ncols=True,
    )

    # Registrar información sobre el progreso
    logger.info(f"Total IDs: {len(all_ids)}")
    logger.info(f"Already processed: {len(processed)}")
    logger.info(f"Remaining to fetch: {len(remaining)}")
    logger.info(f"Output JSONL: {jsonl_path}")

    # Si no hay IDs por procesar, salir de la función
    if not remaining:
        logger.success("Nothing to do. All IDs are already processed.")
        return

    # Crear una cola asíncrona para almacenar los IDs a procesar
    queue: asyncio.Queue = asyncio.Queue()
    # Crear un lock para sincronizar escrituras concurrentes
    write_lock = asyncio.Lock()

    # Encolar todos los IDs que falta procesar
    for imdb_id in remaining:
        await queue.put(imdb_id)

    # Configurar un timeout de 40 segundos para cada request HTTP
    timeout = aiohttp.ClientTimeout(total=40)

    # Crear una sesión HTTP asíncrona con el timeout configurado
    async with aiohttp.ClientSession(timeout=timeout) as session:
        # Crear y lanzar múltiples workers en paralelo
        workers = [
            asyncio.create_task(worker(i, session, queue, jsonl_path, write_lock, progress))
            for i in range(CONCURRENCY)
        ]

        # Encolar una señal de terminación (None) para cada worker
        for _ in workers:
            await queue.put(None)

        # Esperar hasta que todos los IDs de la cola hayan sido procesados
        await queue.join()

        # Cancelar cualquier worker que aún esté en ejecución
        for w in workers:
            w.cancel()

    # Cerrar la barra de progreso
    progress.close()

    # Registrar que la descarga completó exitosamente
    logger.success("Download complete.")
    logger.info(f"Results appended to {jsonl_path}")

In [ ]:
# ========== CARGAR LISTA DE IDs Y EJECUTAR EL DESCARGADOR ==========
with open("data/movie_ids.txt") as f:
    # Leer todos los IDs de IMDb desde el archivo, eliminando espacios en blanco
    all_ids = [line.strip() for line in f if line.strip()]

# Run the async downloader
await main(all_ids[:1000], output_file="data/omdb_raw.jsonl")


19:36:51 | INFO | Total IDs: 1000

19:36:51 | INFO | Already processed: 0

19:36:51 | INFO | Remaining to fetch: 1000

19:36:51 | INFO | Output JSONL: data/omdb_raw.jsonl

19:36:51 | INFO | Worker 0: tt0000574

19:36:51 | INFO | Worker 1: tt0002130

19:36:51 | INFO | Worker 2: tt0002423

19:36:51 | INFO | Worker 3: tt0002844

19:36:51 | INFO | Worker 4: tt0003014



19:36:51 | ERROR | [tt0002423] OMDb error: Request limit reached!

19:36:51 | INFO | Worker 2: tt0003037

19:36:51 | ERROR | [tt0002130] OMDb error: Request limit reached!

19:36:51 | INFO | Worker 1: tt0003165

19:36:51 | ERROR | [tt0002844] OMDb error: Request limit reached!

19:36:51 | INFO | Worker 3: tt0003419

19:36:51 | ERROR | [tt0003014] OMDb error: Request limit reached!

19:36:51 | INFO | Worker 4: tt0003643

19:36:51 | ERROR | [tt0000574] OMDb error: Request limit reached!

19:36:51 | INFO | Worker 0: tt0003740



19:36:51 | ERROR | [tt0003037] OMDb error: Request limit reached!

19:36:51 | INFO | Worker 2: tt0003772

19:36:51 | ERROR | [tt0003165] OMDb error: Request limit reached!

19:36:51 | INFO | Worker 1: tt0003930

19:36:51 | ERROR | [tt0003419] OMDb error: Request limit reached!

19:36:51 | INFO | Worker 3: tt0003952

19:36:51 | ERROR | [tt0003643] OMDb error: Request limit reached!

19:36:51 | INFO | Worker 4: tt0004026

19:36:51 | ERROR | [tt0003740] OMDb error: Request limit reached!

19:36:51 | INFO | Worker 0: tt0004181



19:36:52 | ERROR | [tt0003772] OMDb error: Request limit reached!

19:36:52 | INFO | Worker 2: tt0004465

19:36:52 | ERROR | [tt0003930] OMDb error: Request limit reached!

19:36:52 | INFO | Worker 1: tt0004635

19:36:52 | ERROR | [tt0003952] OMDb error: Request limit reached!

19:36:52 | INFO | Worker 3: tt0004707

19:36:52 | ERROR | [tt0004026] OMDb error: Request limit reached!

19:36:52 | INFO | Worker 4: tt0004873

19:36:52 | ERROR | [tt0004181] OMDb error: Request limit reached!

19:36:52 | INFO | Worker 0: tt0004972



19:36:52 | ERROR | [tt0004465] OMDb error: Request limit reached!

19:36:52 | INFO | Worker 2: tt0005060

19:36:52 | ERROR | [tt0004707] OMDb error: Request limit reached!

19:36:52 | INFO | Worker 3: tt0005078

19:36:52 | ERROR | [tt0004873] OMDb error: Request limit reached!

19:36:52 | INFO | Worker 4: tt0005339

19:36:52 | ERROR | [tt0004635] OMDb error: Request limit reached!

19:36:52 | INFO | Worker 1: tt0005916

19:36:52 | ERROR | [tt0004972] OMDb error: Request limit reached!

19:36:52 | INFO | Worker 0: tt0005951



...

19:38:45 | ERROR | [tt0018455] Unexpected error: Session is closed — retrying in 8s

19:38:45 | ERROR | [tt0018471] Unexpected error: Session is closed — retrying in 8s

19:38:45 | ERROR | [tt0018505] Unexpected error: Session is closed — retrying in 8s

19:38:45 | ERROR | [tt0018515] Unexpected error: Session is closed — retrying in 8s

19:38:45 | ERROR | [tt0018523] Unexpected error: Session is closed — retrying in 8s



19:38:53 | ERROR | [tt0018455] Unexpected error: Session is closed — retrying in 10s

19:38:53 | ERROR | [tt0018471] Unexpected error: Session is closed — retrying in 10s

19:38:53 | ERROR | [tt0018505] Unexpected error: Session is closed — retrying in 10s

19:38:53 | ERROR | [tt0018515] Unexpected error: Session is closed — retrying in 10s

19:38:53 | ERROR | [tt0018523] Unexpected error: Session is closed — retrying in 10s



19:39:03 | CRITICAL | [tt0018455] FAILED permanently after 5 retries

19:39:03 | INFO | Worker 0: tt0018526

19:39:03 | ERROR | [tt0018526] Unexpected error: Session is closed — retrying in 2s

19:39:03 | CRITICAL | [tt0018471] FAILED permanently after 5 retries

19:39:03 | INFO | Worker 2: tt0018528

19:39:03 | ERROR | [tt0018528] Unexpected error: Session is closed — retrying in 2s

19:39:03 | CRITICAL | [tt0018505] FAILED permanently after 5 retries

19:39:03 | INFO | Worker 3: tt0018578

19:39:03 | ERROR | [tt0018578] Unexpected error: Session is closed — retrying in 2s

19:39:03 | CRITICAL | [tt0018515] FAILED permanently after 5 retries

19:39:03 | INFO | Worker 1: tt0018618

19:39:03 | ERROR | [tt0018618] Unexpected error: Session is closed — retrying in 2s

19:39:03 | CRITICAL | [tt0018523] FAILED permanently after 5 retries

19:39:03 | INFO | Worker 4: tt0018684

19:39:03 | ERROR | [tt0018684] Unexpected error: Session is closed — retrying in 2s



19:39:05 | ERROR | [tt0018526] Unexpected error: Session is closed — retrying in 4s

19:39:05 | ERROR | [tt0018528] Unexpected error: Session is closed — retrying in 4s

19:39:05 | ERROR | [tt0018578] Unexpected error: Session is closed — retrying in 4s

19:39:05 | ERROR | [tt0018618] Unexpected error: Session is closed — retrying in 4s

19:39:05 | ERROR | [tt0018684] Unexpected error: Session is closed — retrying in 4s



19:39:09 | ERROR | [tt0018526] Unexpected error: Session is closed — retrying in 6s

19:39:09 | ERROR | [tt0018528] Unexpected error: Session is closed — retrying in 6s

19:39:09 | ERROR | [tt0018578] Unexpected error: Session is closed — retrying in 6s

19:39:09 | ERROR | [tt0018618] Unexpected error: Session is closed — retrying in 6s

19:39:09 | ERROR | [tt0018684] Unexpected error: Session is closed — retrying in 6s

